<a href="https://colab.research.google.com/github/omsai1682/RL_51682/blob/main/Bellman%20Optimality%20%20lab-10_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# ============================================================
# ASSIGNMENT 1: BELLMAN OPTIMALITY AND VALUE ITERATION
# Deterministic 2x2 Grid World
# ============================================================

import pandas as pd

# 1. DEFINE THE MDP

states = ['S0', 'S1', 'G']
actions = ['Up', 'Down', 'Left', 'Right']

gamma = 0.9
theta = 0.000001

rewards = {
    'normal': -1,
    'wall': -1,
    'goal': 10
}

terminal_state = 'G'

# Grid representation:
#
#       S0     Wall
#       S1      G
#
# Hitting a wall or boundary keeps the agent
# in its current state.

transitions = {
    'S0': {
        'Up': ('S0', rewards['wall']),
        'Down': ('S1', rewards['normal']),
        'Left': ('S0', rewards['wall']),
        'Right': ('S0', rewards['wall'])
    },

    'S1': {
        'Up': ('S0', rewards['normal']),
        'Down': ('S1', rewards['wall']),
        'Left': ('S1', rewards['wall']),
        'Right': ('G', rewards['goal'])
    }
}


# 2. DISPLAY MDP INFORMATION

print("=" * 60)
print("1. MDP INFORMATION")
print("=" * 60)

print("States:", states)
print("Actions:", actions)
print("Discount Factor:", gamma)
print("Normal Cell Reward:", rewards['normal'])
print("Wall Reward:", rewards['wall'])
print("Goal Reward:", rewards['goal'])
print("Terminal State:", terminal_state)


# 3. VERIFY ALL TRANSITIONS

print("\n" + "=" * 60)
print("2. TRANSITION VERIFICATION")
print("=" * 60)

for state in states:
    if state == terminal_state:
        continue

    for action in actions:
        next_state, reward = transitions[state][action]

        print(
            f"Current State: {state:2} | "
            f"Action: {action:5} | "
            f"Next State: {next_state:2} | "
            f"Reward: {reward}"
        )

# Verify that every state-action pair is defined
for state in states:
    if state != terminal_state:
        assert set(transitions[state].keys()) == set(actions)

print("\nAll state-action transitions are defined.")


# 4. INITIALIZE STATE VALUES

print("\n" + "=" * 60)
print("3. INITIAL STATE VALUES")
print("=" * 60)

V = {
    'S0': 0.0,
    'S1': 0.0,
    'G': 0.0
}

print(V)


# 5. IMPLEMENT SYNCHRONOUS VALUE ITERATION

print("\n" + "=" * 60)
print("4. VALUE ITERATION")
print("=" * 60)

iteration = 0
history = []

while True:

    iteration += 1

    # Store previous iteration values
    old_V = V.copy()

    # Create new values separately (synchronous update)
    new_V = old_V.copy()

    for state in states:

        if state == terminal_state:
            new_V[state] = 0.0
            continue

        action_values = []

        for action in actions:

            next_state, reward = transitions[state][action]

            q_value = reward + gamma * old_V[next_state]

            action_values.append(q_value)

        new_V[state] = max(action_values)

    # Calculate maximum change
    delta = max(
        abs(new_V[state] - old_V[state])
        for state in states
    )

    # Update values after calculating all states
    V = new_V.copy()

    history.append({
        'Iteration': iteration,
        'V(S0)': V['S0'],
        'V(S1)': V['S1'],
        'V(G)': V['G'],
        'Delta': delta
    })

    print(
        f"Iteration {iteration:2}: "
        f"S0 = {V['S0']:10.6f}, "
        f"S1 = {V['S1']:10.6f}, "
        f"G = {V['G']:10.6f}, "
        f"Delta = {delta:.6f}"
    )

    # Stop when convergence is reached
    if delta < theta:
        break


# Display iteration table

print("\nITERATION-WISE STATE VALUE TABLE")

iteration_df = pd.DataFrame(history)

print(iteration_df.to_string(index=False))


# 6. DISPLAY FINAL OPTIMAL STATE VALUES

print("\n" + "=" * 60)
print("5. FINAL OPTIMAL STATE VALUES")
print("=" * 60)

for state in states:
    print(f"V*({state}) = {V[state]:.6f}")


# 7. CALCULATE ACTION VALUES Q*(s,a)

print("\n" + "=" * 60)
print("6. ACTION VALUE FUNCTION Q*(s,a)")
print("=" * 60)

Q = {}

for state in states:

    if state == terminal_state:
        continue

    Q[state] = {}

    for action in actions:

        next_state, reward = transitions[state][action]

        Q[state][action] = (
            reward + gamma * V[next_state]
        )

        print(
            f"Q*({state}, {action}) = "
            f"{Q[state][action]:.6f}"
        )


# Display Q-value table

print("\nACTION VALUE TABLE")

q_df = pd.DataFrame(Q).T

print(q_df.to_string())


# 8. EXTRACT OPTIMAL POLICY

print("\n" + "=" * 60)
print("7. OPTIMAL POLICY")
print("=" * 60)

optimal_policy = {}

for state in states:

    if state == terminal_state:
        continue

    max_q = max(Q[state].values())

    best_actions = [
        action
        for action, value in Q[state].items()
        if abs(value - max_q) < 1e-9
    ]

    optimal_policy[state] = best_actions

    print(
        f"State {state}: "
        f"Optimal Action(s) = {best_actions}, "
        f"Maximum Q-value = {max_q:.6f}"
    )


# 9. BELLMAN OPTIMALITY VERIFICATION

print("\n" + "=" * 60)
print("8. BELLMAN OPTIMALITY VERIFICATION")
print("=" * 60)

for state in states:

    if state == terminal_state:
        continue

    max_q = max(Q[state].values())

    print(
        f"State: {state} | "
        f"V*(s) = {V[state]:.6f} | "
        f"Max Q*(s,a) = {max_q:.6f} | "
        f"Verified = {abs(V[state] - max_q) < 1e-9}"
    )


# 10. FINAL SUMMARY

print("\n" + "=" * 60)
print("9. FINAL SUMMARY")
print("=" * 60)

print("Convergence Iterations:", iteration)
print("Convergence Threshold:", theta)
print("Discount Factor:", gamma)

print("\nOptimal State Values:")
for state in states:
    print(f"{state}: {V[state]:.6f}")

print("\nOptimal Policy:")
for state, action in optimal_policy.items():
    print(f"{state} -> {', '.join(action)}")

print("\nValue Iteration Completed Successfully!")


1. MDP INFORMATION
States: ['S0', 'S1', 'G']
Actions: ['Up', 'Down', 'Left', 'Right']
Discount Factor: 0.9
Normal Cell Reward: -1
Wall Reward: -1
Goal Reward: 10
Terminal State: G

2. TRANSITION VERIFICATION
Current State: S0 | Action: Up    | Next State: S0 | Reward: -1
Current State: S0 | Action: Down  | Next State: S1 | Reward: -1
Current State: S0 | Action: Left  | Next State: S0 | Reward: -1
Current State: S0 | Action: Right | Next State: S0 | Reward: -1
Current State: S1 | Action: Up    | Next State: S0 | Reward: -1
Current State: S1 | Action: Down  | Next State: S1 | Reward: -1
Current State: S1 | Action: Left  | Next State: S1 | Reward: -1
Current State: S1 | Action: Right | Next State: G  | Reward: 10

All state-action transitions are defined.

3. INITIAL STATE VALUES
{'S0': 0.0, 'S1': 0.0, 'G': 0.0}

4. VALUE ITERATION
Iteration  1: S0 =  -1.000000, S1 =  10.000000, G =   0.000000, Delta = 10.000000
Iteration  2: S0 =   8.000000, S1 =  10.000000, G =   0.000000, Delta = 9.000